# Class 1 & 2: NLP and Search
## Learning Notebook Part 2 - Advanced Search & Clustering

**Welcome to Part 2!** In this notebook, we'll build on the foundation from Part 1:

**You should have completed Part 1** where you learned:
- Keyword search (simple and multiple keyword search)
- Simple tokenization
- Text preprocessing (cleaning, tokenization, regex)
- Bag of Words (word counts - converting text to numbers)
- Understanding vector representations

**Now in Part 2, you'll learn:**
- 📊 **TF-IDF**: Improved text representation (concept - you'll implement in exercises!)
- 🎯 **Similarity-Based Search**: Finding relevant documents using TF-IDF + cosine similarity
- 📦 **Clustering**: Automatically organizing documents with K-Means

**After Part 2**: Continue with **Learning Notebook Part 3** to learn industry tools (NLTK, spaCy, scikit-learn) that consolidate everything you've learned!

Let's dive in!


## Setup


In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# For better output display
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"


### Load the Data


In [ ]:
# Load movie descriptions
# If running in Google Colab and data file doesn't exist, download it from GitHub
import os

if not os.path.exists('data/movies.csv'):
    print("Data file not found. Downloading from GitHub...")
    os.makedirs('data', exist_ok=True)
    import urllib.request
    url = 'https://raw.githubusercontent.com/samsung-ai-course/8th-9th-edition/main/Chapter%202%20-%20Natural%20Language%20Processing/Class%201%20%26%202%20-%20NLP%20and%20Search/data/movies.csv'
    urllib.request.urlretrieve(url, 'data/movies.csv')
    print("✓ Data file downloaded successfully!")

df = pd.read_csv('data/movies.csv')
print(f"Loaded {len(df)} movies")
df.head()

## Sparse vs Dense Vectors (Quick Reminder)

Before we dive into TF-IDF, let's quickly visualize sparse vs dense vectors:


In [ ]:
# Example: Sparse vector (TF-IDF will create this)
# Let's use a small example first
sample_docs = df['description'].head(3).tolist()

# Create TF-IDF vectors (sparse!)
vectorizer = TfidfVectorizer(max_features=50, stop_words='english')
tfidf_matrix = vectorizer.fit_transform(sample_docs)

print("Sparse TF-IDF Matrix Shape:", tfidf_matrix.shape)
print(f"Total elements: {tfidf_matrix.shape[0] * tfidf_matrix.shape[1]}")
print(f"Non-zero elements: {tfidf_matrix.nnz}")
print(f"Sparsity: {(1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100:.2f}%")
print("\nFirst document vector (first 20 values):")
print(tfidf_matrix[0].toarray()[0][:20])

# Compare to dense vector (random example)
dense_example = np.random.rand(50)
print(f"\n\nDense vector (50 dimensions, all non-zero):")
print(dense_example[:20])


**Key Takeaway**: 
- **Sparse vectors** (BoW, TF-IDF) are interpretable - we know what each dimension means
  - **BoW** (from Part 1): Word counts (e.g., "python" appears 3 times)
  - **TF-IDF** (this part): Word frequencies weighted by importance
- **Dense vectors** (embeddings) are more powerful - they capture meaning and relationships
- For now, we'll use **TF-IDF** (sparse) as it's simple and interpretable. Next class, we'll see dense embeddings!


## TF-IDF: Term Frequency-Inverse Document Frequency

**TF-IDF** is an improvement over simple Bag of Words. It weighs words by:
- **TF (Term Frequency)**: How often a word appears in a document (higher = more important to that document)
- **IDF (Inverse Document Frequency)**: How rare a word is across all documents (rare words are more informative)

### The TF-IDF Formula (Step by Step)

**TF-IDF = TF × IDF**

Let's break this down:

#### 1. Term Frequency (TF)

**TF** measures how frequently a term appears in a document. There are several ways to calculate it:

**Raw Count (Simple):**
```
TF(t, d) = count(t, d)
```
- `t` = term (word)
- `d` = document
- Simply counts how many times term `t` appears in document `d`

**Normalized TF (More Common):**
```
TF(t, d) = count(t, d) / total_words_in_document(d)
```
- Normalizes by document length
- Prevents longer documents from having higher scores just because they have more words

**Log-Scaled TF (Alternative):**
```
TF(t, d) = 1 + log(count(t, d))
```
- Uses logarithm to dampen the effect of very frequent terms
- Common in production systems

**Example:**
```
Document: "The space adventure movie is about space exploration"
- "space" appears 2 times
- Total words: 8
- TF("space", document) = 2/8 = 0.25 (normalized)
```

#### 2. Inverse Document Frequency (IDF)

**IDF** measures how rare or common a term is across the entire corpus (collection of documents). Rare words are more informative!

**Standard IDF Formula:**
```
IDF(t, D) = log(N / df(t, D))
```
- `N` = total number of documents in the corpus
- `df(t, D)` = document frequency (number of documents containing term `t`)
- `log` = natural logarithm (or log base 10)

**Why the logarithm?**
- Prevents very rare words from having extremely high IDF values
- Smooths the scaling

**Smooth IDF (Add-1 Smoothing - More Common):**
```
IDF(t, D) = log((N + 1) / (df(t, D) + 1)) + 1
```
- Prevents division by zero if a term doesn't appear in any document
- Adds 1 to avoid taking log of 0
- This is what scikit-learn uses by default!

**Example:**
```
Corpus: 1000 documents
- "the" appears in 950 documents → IDF = log(1000/950) ≈ 0.05 (very low - common word)
- "space" appears in 50 documents → IDF = log(1000/50) ≈ 3.0 (high - rare word)
- "python" appears in 10 documents → IDF = log(1000/10) ≈ 4.6 (very high - very rare)
```

#### 3. TF-IDF (Combined)

**Final TF-IDF Score:**
```
TF-IDF(t, d, D) = TF(t, d) × IDF(t, D)
```

**What this means:**
- **High TF-IDF**: Term appears frequently in the document AND is rare across the corpus
- **Low TF-IDF**: Term is common everywhere (like "the", "a") OR doesn't appear in the document

**Example Calculation:**
```
Document: "The space adventure movie is about space exploration"
Corpus: 1000 documents

For term "space":
- TF("space", document) = 2/8 = 0.25
- df("space", corpus) = 50 (appears in 50 documents)
- IDF("space", corpus) = log(1000/50) ≈ 3.0
- TF-IDF("space", document) = 0.25 × 3.0 = 0.75

For term "the":
- TF("the", document) = 1/8 = 0.125
- df("the", corpus) = 950 (appears in 950 documents)
- IDF("the", corpus) = log(1000/950) ≈ 0.05
- TF-IDF("the", document) = 0.125 × 0.05 = 0.00625 (very low!)
```

**Why it works**: 
- Common words like "the", "a" have high TF but **very low IDF** (they appear in many documents), so their TF-IDF is low
- Important words like "Python", "space", "adventure" have **high TF-IDF** because they're both frequent in the document AND rare across the corpus
- This automatically filters out stop words and highlights important content words!

### TF-IDF Vector Representation

Each document becomes a vector where:
- **Each dimension** = one word in the vocabulary
- **Each value** = TF-IDF score for that word in that document
- **Most values are 0** (sparse vector) - most words don't appear in each document

**Example:**
```
Vocabulary: ["adventure", "exploration", "movie", "space", "the", ...]

Document 1: "space adventure"
→ Vector: [0.5, 0.0, 0.0, 0.7, 0.0, ...]

Document 2: "movie exploration"
→ Vector: [0.0, 0.6, 0.4, 0.0, 0.0, ...]
```

**Important**: You'll implement TF-IDF from scratch in the **Exercise Notebook**! Here we'll just create TF-IDF vectors together (using scikit-learn) so we can use them for similarity search and clustering. The exercise will teach you how TF-IDF actually works step-by-step.


In [ ]:
# TF-IDF Vectors - Let's create them together using scikit-learn!
# NOTE: You'll implement TF-IDF from scratch in Exercise 3!
# Here we'll use scikit-learn to create TF-IDF vectors for similarity search and clustering

# TODO (Together): Create TF-IDF vectorizer
# What parameters should we use?
vectorizer = TfidfVectorizer(max_features=100, stop_words='english', lowercase=True)  # TODO: Discuss these parameters

# TODO (Together): Fit and transform all documents
# What does 'fit' do? What does 'transform' do?
tfidf_vectors = vectorizer.fit_transform(df['description'])  # TODO: Complete this

print(f"TF-IDF Matrix Shape: {tfidf_vectors.shape}")
print(f"(Number of documents, Vocabulary size)")
print(f"\nVocabulary (first 20 words):")
print(vectorizer.get_feature_names_out()[:20])

print(f"\n💡 Remember: You'll learn to implement TF-IDF step-by-step in Exercise 3!")
print(f"   For now, we're using scikit-learn to create vectors for similarity search and clustering.")


## Similarity-Based Search with TF-IDF

**Now let's implement similarity-based search together!** This finds relevant documents based on similarity, not just exact keyword matches.

The idea:
1. Convert query to TF-IDF vector (using the same vectorizer)
2. Compare with all document vectors using **Cosine Similarity**
3. Return most similar documents (highest similarity scores)

**Cosine Similarity**: Measures the angle between two vectors. Range: -1 to 1
- **1** = identical direction (very similar)
- **0** = perpendicular (no similarity)
- **-1** = opposite direction (very different)

**Why cosine?** It measures similarity regardless of document length!


In [ ]:
# Similarity-Based Search - Let's implement this together!
def search_tfidf(query, vectorizer, tfidf_vectors, df, top_k=5):
    """
    Similarity-based search using TF-IDF and cosine similarity
    
    Steps:
    1. Convert query to TF-IDF vector
    2. Calculate cosine similarity with all documents
    3. Get top_k most similar documents
    4. Return results as DataFrame
    """
    # TODO (Together): Step 1 - Convert query to TF-IDF vector using the same vectorizer
    # What method should we use? transform() or fit_transform()?
    query_vector = vectorizer.transform([query])  # TODO: Complete this
    
    # TODO (Together): Step 2 - Calculate cosine similarity with all document vectors
    # How do we compare query_vector with tfidf_vectors?
    similarities = cosine_similarity(query_vector, tfidf_vectors)[0]  # TODO: Complete this
    
    # TODO (Together): Step 3 - Get indices of top_k most similar documents
    # How do we find the indices of the highest similarity scores?
    # Hint: Use argsort() and reverse order
    top_indices = similarities.argsort()[-top_k:][::-1]  # TODO: Complete this
    
    # TODO (Together): Step 4 - Build results DataFrame
    results = []
    for idx in top_indices:
        results.append({
            'movie_id': df.iloc[idx]['movie_id'],
            'title': df.iloc[idx]['title'],
            'similarity': similarities[idx],
            'description': df.iloc[idx]['description']
        })
    
    return pd.DataFrame(results)

# Let's test it!
example_query = "space exploration adventure"
print(f"Query: '{example_query}'")
print("\nResults:")
results = search_tfidf(example_query, vectorizer, tfidf_vectors, df, top_k=5)
print(results[['title', 'similarity']])


In [ ]:
# Compare Cosine vs Euclidean: Quantitative Comparison
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

test_query = "space exploration adventure"

# Get results from both methods
results_cosine = search_tfidf(test_query, vectorizer, tfidf_vectors, df, top_k=10)
results_euclidean = search_euclidean(test_query, vectorizer, tfidf_vectors, df, top_k=10)

print("=" * 80)
print(f"COMPARISON: Cosine Similarity vs Euclidean Distance")
print(f"Query: '{test_query}'")
print("=" * 80)

# Side-by-side comparison of top 10 results
print("\n" + "=" * 80)
print("Top 10 Results: Cosine Similarity")
print("=" * 80)
print(results_cosine[['title', 'similarity']].to_string(index=False))

print("\n" + "=" * 80)
print("Top 10 Results: Euclidean Distance")
print("=" * 80)
print(results_euclidean[['title', 'distance']].to_string(index=False))

# Calculate overlap
cosine_titles = set(results_cosine['title'].tolist())
euclidean_titles = set(results_euclidean['title'].tolist())
overlap = cosine_titles.intersection(euclidean_titles)
overlap_pct = (len(overlap) / 10) * 100

print("\n" + "=" * 80)
print("Overlap Analysis")
print("=" * 80)
print(f"Movies in both top-10 lists: {len(overlap)} / 10 ({overlap_pct:.0f}%)")
print(f"Overlapping movies: {', '.join(list(overlap)[:5])}{'...' if len(overlap) > 5 else ''}")
print(f"\nDifferences:")
print(f"  Only in Cosine top-10: {len(cosine_titles - euclidean_titles)} movies")
print(f"  Only in Euclidean top-10: {len(euclidean_titles - cosine_titles)} movies")

# Visualization: Show both metric results in 2D space
print("\n" + "=" * 80)
print("Visual Comparison: Plotting top results from both metrics")
print("=" * 80)

# Get query vector and all vectors
query_vector = vectorizer.transform([test_query])
from scipy.sparse import vstack
all_vecs_for_viz = vstack([query_vector, tfidf_vectors])

# Apply PCA for 2D visualization
pca = PCA(n_components=2, random_state=42)
all_vecs_dense = all_vecs_for_viz.toarray()
vectors_2d = pca.fit_transform(all_vecs_dense)

query_2d = vectors_2d[0]  # First point is the query
docs_2d = vectors_2d[1:]  # Rest are documents

# Get indices of top results from both methods
cosine_indices = [df[df['title'] == title].index[0] for title in results_cosine['title'][:5]]
euclidean_indices = [df[df['title'] == title].index[0] for title in results_euclidean['title'][:5]]

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Plot 1: Cosine Similarity Results
ax1.scatter(docs_2d[:, 0], docs_2d[:, 1], c='lightgray', alpha=0.3, s=20, label='All documents')
ax1.scatter(query_2d[0], query_2d[1], c='red', s=300, marker='*', 
           edgecolors='black', linewidths=2, label='Query', zorder=10)
ax1.scatter(docs_2d[cosine_indices, 0], docs_2d[cosine_indices, 1], 
           c='blue', s=150, alpha=0.7, edgecolors='black', linewidths=1.5,
           label='Top-5 (Cosine)', zorder=5)

# Add labels for top-5
for i, idx in enumerate(cosine_indices[:5]):
    ax1.annotate(f"C{i+1}", (docs_2d[idx, 0], docs_2d[idx, 1]),
                xytext=(5, 5), textcoords='offset points',
                fontsize=9, fontweight='bold')

ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax1.set_title('Cosine Similarity: Top-5 Results', fontweight='bold', fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Euclidean Distance Results
ax2.scatter(docs_2d[:, 0], docs_2d[:, 1], c='lightgray', alpha=0.3, s=20, label='All documents')
ax2.scatter(query_2d[0], query_2d[1], c='red', s=300, marker='*',
           edgecolors='black', linewidths=2, label='Query', zorder=10)
ax2.scatter(docs_2d[euclidean_indices, 0], docs_2d[euclidean_indices, 1],
           c='green', s=150, alpha=0.7, edgecolors='black', linewidths=1.5,
           label='Top-5 (Euclidean)', zorder=5)

# Add labels for top-5
for i, idx in enumerate(euclidean_indices[:5]):
    ax2.annotate(f"E{i+1}", (docs_2d[idx, 0], docs_2d[idx, 1]),
                xytext=(5, 5), textcoords='offset points',
                fontsize=9, fontweight='bold')

ax2.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax2.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax2.set_title('Euclidean Distance: Top-5 Results', fontweight='bold', fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Key Observations:")
print("  - Both plots show the SAME vector space, just highlighting different results")
print("  - Cosine considers DIRECTION (angle) - documents with similar word patterns")
print("  - Euclidean considers DISTANCE (position) - documents close in absolute space")
print("  - Red star (*) = your query")
print("  - Blue (C1-C5) = Cosine top-5 results")
print("  - Green (E1-E5) = Euclidean top-5 results")
print("\n🎯 Understanding the Difference:")
print("  - Cosine: Good for text because it ignores document length (magnitude)")
print("  - Euclidean: Sensitive to magnitude - long documents may have higher distances")
print("  - For TF-IDF text search, Cosine is usually better!")
print("  - But both can work - it depends on your use case!")

## Euclidean Distance: Another Way to Measure Similarity

We just learned about **Cosine Similarity**, which measures the angle between vectors. Now let's learn about **Euclidean Distance**, which measures the straight-line distance between vectors!

**Euclidean Distance**: The straight-line distance between two points in space.
- **Range**: 0 to ∞ (infinity)
- **0** = identical (same point)
- **Larger values** = more different (farther apart)

**Formula** (for 2D, but works for any dimension):
```
distance = √[(x₁ - x₂)² + (y₁ - y₂)²]
```

**Key Difference from Cosine**:
- **Cosine Similarity**: Measures direction/angle (ignores magnitude/length)
  - Example: [1, 2] and [2, 4] have cosine = 1.0 (same direction)
- **Euclidean Distance**: Measures absolute distance (considers magnitude)
  - Example: [1, 2] and [2, 4] have distance > 0 (different positions)

**When to use which?**
- **Cosine**: When you care about direction/pattern, not magnitude (text similarity, document comparison)
- **Euclidean**: When you care about absolute distance (spatial data, measurements)

For text/TF-IDF vectors:
- **Cosine is usually better** because document length doesn't matter (long documents vs short documents)
- **But Euclidean can work too!** Let's compare them!

**Important**: For Euclidean distance with search results, we want **smaller = more similar**. So we'll sort by smallest distance!

### Comparing Cosine vs Euclidean: Side-by-Side

Let's compare the two distance metrics directly to understand when results differ and why!

In [ ]:
# Euclidean Distance Search - Let's implement this together!
from sklearn.metrics.pairwise import euclidean_distances

def search_euclidean(query, vectorizer, tfidf_vectors, df, top_k=5):
    """
    Similarity-based search using TF-IDF and Euclidean distance
    
    Steps:
    1. Convert query to TF-IDF vector
    2. Calculate Euclidean distance with all documents
    3. Get top_k closest documents (smallest distance = most similar!)
    4. Return results as DataFrame
    """
    # TODO (Together): Step 1 - Convert query to TF-IDF vector
    query_vector = vectorizer.transform([query])
    
    # TODO (Together): Step 2 - Calculate Euclidean distance with all document vectors
    # Note: euclidean_distances returns a matrix, we want the first row
    distances = euclidean_distances(query_vector, tfidf_vectors)[0]
    
    # TODO (Together): Step 3 - Get indices of top_k SMALLEST distances
    # For Euclidean: smaller = more similar!
    # Hint: argsort() gives indices in ascending order (smallest first)
    top_indices = distances.argsort()[:top_k]  # Already in ascending order!
    
    # TODO (Together): Step 4 - Build results DataFrame
    results = []
    for idx in top_indices:
        results.append({
            'movie_id': df.iloc[idx]['movie_id'],
            'title': df.iloc[idx]['title'],
            'distance': distances[idx],
            'description': df.iloc[idx]['description']
        })
    
    return pd.DataFrame(results)

# Let's test it with the same query!
example_query = "space exploration adventure"
print(f"Query: '{example_query}'")
print("\nResults (Euclidean Distance):")
results_euclidean = search_euclidean(example_query, vectorizer, tfidf_vectors, df, top_k=5)
print(results_euclidean[['title', 'distance']])

print("\n💡 Note: Smaller distance = more similar!")
print("   Compare these results with cosine similarity results above.")

### Compare: Keyword Search vs Similarity-Based Search (TF-IDF)

Let's see how they differ:

**Note**: TF-IDF similarity search is better than simple keyword matching, but it's still fundamentally keyword-based (not true semantic search). True semantic search that understands synonyms and meaning requires embeddings (Class 3)!


In [ ]:
# Comparison: Keyword Search vs Similarity-Based Search
# You'll implement both in exercises!

print("=" * 60)
print("KEY DIFFERENCES: Keyword Search vs Similarity Search")
print("=" * 60)

print("\nKeyword Search (from Part 1 - Exercise 4):")
print("  ✅ Fast and simple")
print("  ✅ Finds exact word matches")
print("  ✅ Multiple keyword search allows more specific queries")
print("  ❌ No ranking - all matches are equal")
print("  ❌ 'mind-bending' won't find 'psychological thriller'")
print("  ❌ Limited to exact words")

print("\nSimilarity-Based Search (TF-IDF - Exercise 5):")
print("  ✅ Ranks by importance (TF-IDF weighted)")
print("  ✅ Better than keyword search")
print("  ✅ Finds documents with similar word patterns")
print("  ⚠️ Still keyword-based (not true semantic)")
print("  ⚠️ 'mind-bending' might match 'psychological' IF they share other words")
print("  ❌ Still cannot understand synonyms or true meaning")

print("\nTrue Semantic Search (Embeddings - Class 3!):")
print("  ✅ Understands meaning and synonyms")
print("  ✅ 'space' and 'cosmic' are similar (semantic similarity)")
print("  ✅ True understanding of meaning")
print("  ⚠️ Requires embeddings (Class 3)")

print("\n💡 Key Point: TF-IDF is BETTER than keyword search,")
print("   but it's still SYNTAX-based (word patterns), not SEMANTIC (meaning).")
print("   Semantic = meaning. True semantic search requires embeddings!")

print("\n💡 Implementation:")
print("   - Exercise 4: Keyword search")
print("   - Exercise 5: Similarity-based search with TF-IDF")
print("   - Class 3: Semantic search with embeddings")


## Hybrid Search: Combining TF-IDF with Keyword Matching

**Real-world search systems** often combine multiple approaches to get the best results. Let's build a **hybrid search** that combines:
1. **TF-IDF similarity search** (semantic-like matching, ranks by importance)
2. **Keyword matching** (exact word matches, boosts precision)

**Why Hybrid?**
- **TF-IDF alone**: Might miss exact matches or rank them lower
- **Keyword alone**: Too rigid, misses related content
- **Hybrid**: Best of both worlds - precision from keywords + recall from TF-IDF!

### Hybrid Search Strategy

**Approach**: Combine scores from both methods
1. Get TF-IDF similarity scores (0 to 1)
2. Get keyword match scores (binary: 0 or 1, or count-based)
3. Combine with weighted sum: `final_score = α × keyword_score + (1-α) × tfidf_score`
   - `α` (alpha) = weight for keyword matching (e.g., 0.3 = 30% keyword, 70% TF-IDF)

**Alternative**: Boost exact matches
- If query words appear in document → boost TF-IDF score
- Simple: `final_score = tfidf_score + boost × keyword_match_count`

Let's implement a hybrid search for our movie search system!


In [ ]:
# Hybrid Search Implementation - Combining TF-IDF + Keyword Matching
def hybrid_search(query, vectorizer, tfidf_vectors, df, top_k=5, keyword_weight=0.3, boost_factor=0.2):
    """
    Hybrid search combining TF-IDF similarity and keyword matching
    
    Args:
        query: Search query string
        vectorizer: Fitted TF-IDF vectorizer
        tfidf_vectors: TF-IDF vectors for all documents
        df: DataFrame with movie data
        top_k: Number of results to return
        keyword_weight: Weight for keyword matching (0-1). 0.3 = 30% keyword, 70% TF-IDF
        boost_factor: Additional boost for exact keyword matches (added to TF-IDF score)
    
    Returns:
        DataFrame with search results sorted by combined score
    """
    import re
    
    # Step 1: Get TF-IDF similarity scores
    query_vector = vectorizer.transform([query])
    tfidf_similarities = cosine_similarity(query_vector, tfidf_vectors)[0]
    
    # Step 2: Get keyword match scores
    query_words = set(re.findall(r'\w+', query.lower()))  # Extract words from query
    keyword_scores = []
    keyword_match_counts = []
    
    for idx, row in df.iterrows():
        text = str(row['description']).lower()
        text_words = set(re.findall(r'\w+', text))
        
        # Count how many query words appear in document
        matches = query_words.intersection(text_words)
        match_count = len(matches)
        match_ratio = match_count / len(query_words) if len(query_words) > 0 else 0
        
        keyword_scores.append(match_ratio)  # Normalized: 0 to 1
        keyword_match_counts.append(match_count)  # Raw count
    
    keyword_scores = np.array(keyword_scores)
    keyword_match_counts = np.array(keyword_match_counts)
    
    # Step 3: Combine scores using weighted sum
    # Normalize TF-IDF scores to 0-1 range (they're already in that range, but ensure it)
    tfidf_normalized = (tfidf_similarities - tfidf_similarities.min()) / (tfidf_similarities.max() - tfidf_similarities.min() + 1e-8)
    
    # Weighted combination
    combined_scores = (keyword_weight * keyword_scores) + ((1 - keyword_weight) * tfidf_normalized)
    
    # Step 4: Add boost for exact keyword matches
    # Documents with more keyword matches get a boost
    max_matches = keyword_match_counts.max() if keyword_match_counts.max() > 0 else 1
    keyword_boost = (keyword_match_counts / max_matches) * boost_factor
    final_scores = combined_scores + keyword_boost
    
    # Step 5: Get top_k results
    top_indices = final_scores.argsort()[-top_k:][::-1]
    
    # Build results
    results = []
    for idx in top_indices:
        results.append({
            'movie_id': df.iloc[idx]['movie_id'],
            'title': df.iloc[idx]['title'],
            'final_score': final_scores[idx],
            'tfidf_score': tfidf_similarities[idx],
            'keyword_score': keyword_scores[idx],
            'keyword_matches': keyword_match_counts[idx],
            'description': df.iloc[idx]['description']
        })
    
    return pd.DataFrame(results)

# Test hybrid search
print("=" * 70)
print("Hybrid Search Example")
print("=" * 70)

query = "space adventure"
print(f"\nQuery: '{query}'")
print(f"\nResults (showing scores breakdown):\n")

results_hybrid = hybrid_search(query, vectorizer, tfidf_vectors, df, top_k=5, 
                               keyword_weight=0.3, boost_factor=0.2)

print(results_hybrid[['title', 'final_score', 'tfidf_score', 'keyword_score', 'keyword_matches']].to_string(index=False))

print("\n" + "=" * 70)
print("Understanding the Scores:")
print("=" * 70)
print("  - final_score: Combined score (keyword + TF-IDF + boost)")
print("  - tfidf_score: TF-IDF cosine similarity (0-1)")
print("  - keyword_score: Ratio of query words found (0-1)")
print("  - keyword_matches: Number of query words that appear in document")
print("\n💡 Documents with exact keyword matches get boosted!")
print("💡 But TF-IDF still helps find related content even without exact matches!")


### Comparing Search Methods: Keyword vs TF-IDF vs Hybrid

Let's see how different search approaches perform on the same query:


In [ ]:
# Compare: Keyword Search vs TF-IDF vs Hybrid Search
def simple_keyword_search_comparison(query, df, top_k=5):
    """Simple keyword search for comparison"""
    import re
    query_words = set(re.findall(r'\w+', query.lower()))
    results = []
    
    for idx, row in df.iterrows():
        text = str(row['description']).lower()
        text_words = set(re.findall(r'\w+', text))
        matches = query_words.intersection(text_words)
        match_count = len(matches)
        
        if match_count > 0:
            results.append({
                'movie_id': row['movie_id'],
                'title': row['title'],
                'match_count': match_count,
                'description': row['description']
            })
    
    # Sort by match count (descending)
    results_df = pd.DataFrame(results)
    if len(results_df) > 0:
        results_df = results_df.sort_values('match_count', ascending=False).head(top_k)
    return results_df

# Test all three methods
test_query = "space adventure"

print("=" * 80)
print(f"COMPARISON: Search Results for Query '{test_query}'")
print("=" * 80)

# Method 1: Keyword Search
print("\n" + "=" * 80)
print("1. KEYWORD SEARCH (Exact Word Matching)")
print("=" * 80)
keyword_results = simple_keyword_search_comparison(test_query, df, top_k=5)
if len(keyword_results) > 0:
    print(keyword_results[['title', 'match_count']].to_string(index=False))
    print(f"\n✅ Finds documents with exact word matches")
    print(f"❌ No ranking beyond match count")
    print(f"❌ Misses related content without exact words")
else:
    print("No results found")

# Method 2: TF-IDF Search
print("\n" + "=" * 80)
print("2. TF-IDF SIMILARITY SEARCH")
print("=" * 80)
tfidf_results = search_tfidf(test_query, vectorizer, tfidf_vectors, df, top_k=5)
print(tfidf_results[['title', 'similarity']].to_string(index=False))
print(f"\n✅ Ranks by importance (TF-IDF weighted)")
print(f"✅ Finds related content even without exact matches")
print(f"⚠️  Might rank exact matches lower if they have low TF-IDF")

# Method 3: Hybrid Search
print("\n" + "=" * 80)
print("3. HYBRID SEARCH (TF-IDF + Keyword Matching)")
print("=" * 80)
hybrid_results = hybrid_search(test_query, vectorizer, tfidf_vectors, df, top_k=5, 
                               keyword_weight=0.3, boost_factor=0.2)
print(hybrid_results[['title', 'final_score', 'tfidf_score', 'keyword_matches']].to_string(index=False))
print(f"\n✅ Combines best of both: keyword precision + TF-IDF recall")
print(f"✅ Boosts exact matches while still finding related content")
print(f"✅ Most flexible and robust approach")

print("\n" + "=" * 80)
print("SUMMARY: When to Use Each Method")
print("=" * 80)
print("""
Keyword Search:
  - ✅ Fast and simple
  - ✅ Good for exact phrase matching
  - ❌ Too rigid, misses related content
  - Use when: You need exact matches only

TF-IDF Search:
  - ✅ Better ranking by importance
  - ✅ Finds related content
  - ⚠️  Might miss exact matches
  - Use when: You want semantic-like matching (but still keyword-based)

Hybrid Search:
  - ✅ Best of both worlds
  - ✅ Precision from keywords + recall from TF-IDF
  - ✅ Most robust for production systems
  - Use when: Building a real search system (recommended!)
""")

print("💡 For our movie search system, hybrid search gives the best results!")


### Tuning Hybrid Search Parameters

**Key Parameters to Adjust:**

1. **`keyword_weight`** (0 to 1):
   - **0.0** = Pure TF-IDF search (no keyword boost)
   - **0.3-0.4** = Balanced (recommended starting point)
   - **0.5-0.7** = More emphasis on exact keyword matches
   - **1.0** = Pure keyword search (no TF-IDF)

2. **`boost_factor`** (0 to 1):
   - **0.0** = No additional boost for keyword matches
   - **0.1-0.3** = Moderate boost (recommended)
   - **0.5+** = Strong boost (may over-prioritize exact matches)

**Tuning Strategy:**
- Start with `keyword_weight=0.3` and `boost_factor=0.2`
- Test on sample queries and check if results make sense
- If exact matches are too low → increase `keyword_weight` or `boost_factor`
- If results are too rigid → decrease `keyword_weight`

**Example: Different Configurations**

```python
# Configuration 1: Balanced (default)
hybrid_search(query, ..., keyword_weight=0.3, boost_factor=0.2)

# Configuration 2: More emphasis on exact matches
hybrid_search(query, ..., keyword_weight=0.5, boost_factor=0.3)

# Configuration 3: More TF-IDF, less keyword
hybrid_search(query, ..., keyword_weight=0.2, boost_factor=0.1)
```

**💡 Pro Tip**: In production, you might want to:
- Use A/B testing to find optimal parameters
- Different weights for different query types (short vs long queries)
- User feedback to continuously improve the search quality


# TODO: Create your own sentences and experiment with different methods!
# Try to create:
# - 2-3 similar sentences (should appear close together in space)
# - 1-2 dissimilar sentences (should appear far from the similar ones)

# Example sentences (modify these or create your own!)
custom_sentences = [
    # Similar sentences (space/adventure theme)
    "space adventure movie with aliens",
    "cosmic journey film about exploration",
    "galactic quest story in outer space",
    
    # Dissimilar sentence (completely different topic)
    "cooking recipe book for desserts",
    # Add more dissimilar sentences here!
]

print("Your custom sentences:")
for i, sent in enumerate(custom_sentences, 1):
    print(f"  {i}. {sent}")

# 🎯 EXPERIMENT: Try different combinations!
# Change these parameters and see what happens:
METHOD = 'tfidf'  # Try: 'tfidf' or 'bow'
USE_PREPROCESSING = True  # Try: True or False

print("\n" + "=" * 70)
print(f"🔬 Experiment: {METHOD.upper()} {'with' if USE_PREPROCESSING else 'without'} preprocessing")
print("=" * 70)

# Visualize them in vector space!
custom_2d, corpus_2d, pca, vec = visualize_sentences_in_space(
    custom_sentences=custom_sentences,
    corpus_texts=df['description'].tolist(),
    method=METHOD,
    use_preprocessing=USE_PREPROCESSING,
    corpus_labels=df['title'].tolist(),
    sample_size=50
)

# Calculate distances between sentences using BOTH metrics!
print("\n" + "=" * 70)
print("Distances Between Your Sentences:")
print("=" * 70)
from scipy.spatial.distance import euclidean, cosine

# Get vectors for all custom sentences
custom_vectors = vec.transform(custom_sentences)

for i in range(len(custom_sentences)):
    for j in range(i+1, len(custom_sentences)):
        # Euclidean distance in 2D (for visualization)
        dist_euclidean_2d = euclidean(custom_2d[i], custom_2d[j])
        
        # Cosine similarity in full space (more accurate)
        vec_i = custom_vectors[i].toarray().flatten()
        vec_j = custom_vectors[j].toarray().flatten()
        # cosine() from scipy returns distance (1 - similarity), so we convert to similarity
        cosine_sim = 1 - cosine(vec_i, vec_j)
        
        # Euclidean distance in full space (more accurate)
        dist_euclidean_full = euclidean(vec_i, vec_j)
        
        print(f"\nSentences {i+1} ↔ {j+1}:")
        print(f"  '{custom_sentences[i]}'")
        print(f"  '{custom_sentences[j]}'")
        print(f"  → Cosine Similarity: {cosine_sim:.3f} (1.0 = identical, 0 = unrelated)")
        print(f"  → Euclidean Distance (full): {dist_euclidean_full:.3f} (0 = identical, larger = more different)")
        print(f"  → Euclidean Distance (2D):  {dist_euclidean_2d:.2f} (for visualization)")

print("\n💡 Observations:")
print("  - Cosine Similarity: Higher values = more similar (direction/angle)")
print("  - Euclidean Distance: Lower values = more similar (absolute distance)")
print("  - Both metrics can give different rankings!")
print("  - 2D distances are for visualization - full-space distances are more accurate")

print("\n" + "=" * 70)
print("🎓 Try This:")
print("=" * 70)
print("1. Run this cell again with METHOD='bow' - how do distances compare?")
print("2. Try USE_PREPROCESSING=False - do sentences cluster differently?")
print("3. Which combination + which metric works best for your sentences?")
print("4. Notice how preprocessing affects BOTH distance metrics!")
print("5. Do cosine and Euclidean agree on which sentences are most similar?")

### Understanding PCA for Visualization

> **📌 Note**: PCA (Principal Component Analysis) is a dimensionality reduction technique used here **purely for visualization purposes**. The details of how PCA works are **out of scope** for this course. You just need to know: it helps us plot high-dimensional vectors in 2D space so we can see patterns visually.

**The Problem**: TF-IDF vectors are high-dimensional (100+ dimensions). We can't visualize 100D space!

**The Solution**: PCA projects high-dimensional vectors to 2D for visualization.

**What you need to know**:
- PCA reduces dimensions while preserving main patterns
- Similar sentences will still appear close together in 2D
- Some information is lost, but key relationships are preserved
- **Variance Explained**: Tells us how much information is preserved (e.g., 80% variance = 80% of information kept)

**Important**: We still use full TF-IDF vectors for search/clustering - PCA is only for visualization!

Let's visualize sentences in 2D space!

In [ ]:
# TODO: Create your query sentence and experiment with different methods!
# Try different queries and see which movies are most similar!

my_query = "space adventure with aliens and exploration"  # Modify this!

# 🎯 EXPERIMENT: Try different vectorization methods!
METHOD = 'tfidf'  # Try: 'tfidf' or 'bow'
USE_PREPROCESSING = True  # Try: True or False

print("=" * 70)
print(f"Finding movies similar to: '{my_query}'")
print(f"Using: {METHOD.upper()} {'with' if USE_PREPROCESSING else 'without'} preprocessing")
print("=" * 70)

# Create vectorizer based on method
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

if METHOD == 'tfidf':
    if USE_PREPROCESSING:
        vec = TfidfVectorizer(max_features=100, stop_words='english', lowercase=True)
    else:
        vec = TfidfVectorizer(max_features=100, lowercase=False)
else:  # bow
    if USE_PREPROCESSING:
        vec = CountVectorizer(max_features=100, stop_words='english', lowercase=True)
    else:
        vec = CountVectorizer(max_features=100, lowercase=False)

# Fit and transform corpus
corpus_vecs = vec.fit_transform(df['description'])
query_vec = vec.transform([my_query])

# Calculate similarities using BOTH metrics
similarities_cosine = cosine_similarity(query_vec, corpus_vecs)[0]
distances_euclidean = euclidean_distances(query_vec, corpus_vecs)[0]

# Get top 5 from BOTH metrics
top_indices_cosine = similarities_cosine.argsort()[-5:][::-1]
top_indices_euclidean = distances_euclidean.argsort()[:5]

similar_movies_cosine = pd.DataFrame({
    'title': [df.iloc[i]['title'] for i in top_indices_cosine],
    'cosine_similarity': [similarities_cosine[i] for i in top_indices_cosine],
    'description': [df.iloc[i]['description'] for i in top_indices_cosine]
})

similar_movies_euclidean = pd.DataFrame({
    'title': [df.iloc[i]['title'] for i in top_indices_euclidean],
    'euclidean_distance': [distances_euclidean[i] for i in top_indices_euclidean],
    'description': [df.iloc[i]['description'] for i in top_indices_euclidean]
})

print("\nTop 5 Most Similar Movies (Cosine Similarity):")
print(similar_movies_cosine[['title', 'cosine_similarity']].to_string(index=False))

print("\nTop 5 Most Similar Movies (Euclidean Distance):")
print(similar_movies_euclidean[['title', 'euclidean_distance']].to_string(index=False))

# Compare overlap
cosine_set = set(similar_movies_cosine['title'])
euclidean_set = set(similar_movies_euclidean['title'])
overlap = cosine_set.intersection(euclidean_set)

print(f"\n📊 Overlap: {len(overlap)}/5 movies appear in both top-5 lists")
if overlap:
    print(f"   Common movies: {', '.join(overlap)}")

# Visualize: Show your query and the most similar movies in space (BOTH metrics!)
print("\n" + "=" * 70)
print("Visualizing your query and similar movies in vector space...")
print("=" * 70)

# Get combined top movies (union of both methods, take top 3 from each)
all_top_descriptions = list(set(
    similar_movies_cosine.head(3)['description'].tolist() + 
    similar_movies_euclidean.head(3)['description'].tolist()
))[:5]  # Limit to 5 total

sentences_to_visualize = [my_query] + all_top_descriptions

# Visualize
custom_2d, corpus_2d, pca, _ = visualize_sentences_in_space(
    custom_sentences=sentences_to_visualize,
    corpus_texts=df['description'].tolist(),
    method=METHOD,
    use_preprocessing=USE_PREPROCESSING,
    corpus_labels=df['title'].tolist(),
    sample_size=50
)

# Show distances using BOTH metrics
print("\n" + "=" * 70)
print("Distances from Your Query (Using BOTH Metrics):")
print("=" * 70)
from scipy.spatial.distance import euclidean, cosine

query_pos = custom_2d[0]  # First sentence is the query

# Show top 3 from cosine
print("\n🔵 Top 3 from Cosine Similarity:")
for i, row in similar_movies_cosine.head(3).iterrows():
    title = row['title']
    cosine_sim = row['cosine_similarity']
    
    # Find in visualization
    try:
        viz_idx = [j for j, desc in enumerate(all_top_descriptions) if desc == row['description']][0] + 1
        movie_pos = custom_2d[viz_idx]
        dist_2d = euclidean(query_pos, movie_pos)
        print(f"\n  {title}:")
        print(f"    Cosine Similarity: {cosine_sim:.3f} (higher = more similar)")
        print(f"    Distance in 2D plot: {dist_2d:.2f}")
    except:
        print(f"\n  {title}:")
        print(f"    Cosine Similarity: {cosine_sim:.3f}")

# Show top 3 from Euclidean
print("\n🟢 Top 3 from Euclidean Distance:")
for i, row in similar_movies_euclidean.head(3).iterrows():
    title = row['title']
    eucl_dist = row['euclidean_distance']
    
    # Find in visualization
    try:
        viz_idx = [j for j, desc in enumerate(all_top_descriptions) if desc == row['description']][0] + 1
        movie_pos = custom_2d[viz_idx]
        dist_2d = euclidean(query_pos, movie_pos)
        print(f"\n  {title}:")
        print(f"    Euclidean Distance: {eucl_dist:.3f} (lower = more similar)")
        print(f"    Distance in 2D plot: {dist_2d:.2f}")
    except:
        print(f"\n  {title}:")
        print(f"    Euclidean Distance: {eucl_dist:.3f}")

print("\n💡 Key Insights:")
print("  - Movies ranked high by BOTH metrics are clearly most relevant!")
print("  - Cosine focuses on word pattern similarity (direction)")
print("  - Euclidean focuses on overall distance in vector space")
print("  - In the plot, closer points should rank higher")
print("  - 2D distances are approximate - full-space metrics are more accurate")

print("\n" + "=" * 70)
print("🎓 Try This:")
print("=" * 70)
print("1. Try METHOD='bow' - do you get different similar movies?")
print("2. Try USE_PREPROCESSING=False - how does it affect results?")
print("3. Which metric finds the most relevant movies for your query?")
print("4. Do both metrics agree? If not, which one makes more sense?")

### Exercise 1: Create Similar and Dissimilar Sentences

**Your Task**: Create sentences that are:
- **Similar** to each other (should appear close in vector space)
- **Dissimilar** to each other (should appear far apart in vector space)

**Tips for creating similar sentences**:
- Use similar words and topics
- Example: "space adventure movie" and "cosmic journey film" (similar topic, different words)

**Tips for creating dissimilar sentences**:
- Use completely different topics
- Example: "space adventure movie" and "cooking recipe book" (completely different)

Let's try it!


In [ ]:
# TODO: Create your own sentences and experiment with different methods!
# Try to create:
# - 2-3 similar sentences (should appear close together in space)
# - 1-2 dissimilar sentences (should appear far from the similar ones)

# Example sentences (modify these or create your own!)
custom_sentences = [
    # Similar sentences (space/adventure theme)
    "space adventure movie with aliens",
    "cosmic journey film about exploration",
    "galactic quest story in outer space",
    
    # Dissimilar sentence (completely different topic)
    "cooking recipe book for desserts",
    # Add more dissimilar sentences here!
]

print("Your custom sentences:")
for i, sent in enumerate(custom_sentences, 1):
    print(f"  {i}. {sent}")

# 🎯 EXPERIMENT: Try different combinations!
# Change these parameters and see what happens:
METHOD = 'tfidf'  # Try: 'tfidf' or 'bow'
USE_PREPROCESSING = True  # Try: True or False

print("\n" + "=" * 70)
print(f"🔬 Experiment: {METHOD.upper()} {'with' if USE_PREPROCESSING else 'without'} preprocessing")
print("=" * 70)

# Visualize them in vector space!
custom_2d, corpus_2d, pca, vec = visualize_sentences_in_space(
    custom_sentences=custom_sentences,
    corpus_texts=df['description'].tolist(),
    method=METHOD,
    use_preprocessing=USE_PREPROCESSING,
    corpus_labels=df['title'].tolist(),
    sample_size=50
)

# Calculate distances between sentences
print("\n" + "=" * 70)
print("Distances Between Your Sentences (in 2D space):")
print("=" * 70)
from scipy.spatial.distance import euclidean

for i in range(len(custom_sentences)):
    for j in range(i+1, len(custom_sentences)):
        dist = euclidean(custom_2d[i], custom_2d[j])
        print(f"\nDistance between:")
        print(f"  '{custom_sentences[i]}'")
        print(f"  '{custom_sentences[j]}'")
        print(f"  → Distance: {dist:.2f} (smaller = more similar)")

print("\n💡 Observations:")
print("  - Similar sentences should have SMALL distances")
print("  - Dissimilar sentences should have LARGE distances")
print("  - Check if your similar sentences are close together!")

print("\n" + "=" * 70)
print("🎓 Try This:")
print("=" * 70)
print("1. Run this cell again with METHOD='bow' - how does it compare?")
print("2. Try USE_PREPROCESSING=False - do sentences cluster differently?")
print("3. Which combination works best for your sentences?")
print("4. Notice how preprocessing affects the vocabulary and clustering!")


### Exercise 2: Find Similar Sentences from the Corpus

**Your Task**: Create a sentence and find the most similar sentences from the movie corpus!

This exercise helps you understand:
- How TF-IDF similarity works in practice
- Which movies are most similar to your query
- How similarity scores relate to vector space positions


In [ ]:
# TODO: Create your query sentence and experiment with different methods!
# Try different queries and see which movies are most similar!

my_query = "space adventure with aliens and exploration"  # Modify this!

# 🎯 EXPERIMENT: Try different vectorization methods!
METHOD = 'tfidf'  # Try: 'tfidf' or 'bow'
USE_PREPROCESSING = True  # Try: True or False

print("=" * 70)
print(f"Finding movies similar to: '{my_query}'")
print(f"Using: {METHOD.upper()} {'with' if USE_PREPROCESSING else 'without'} preprocessing")
print("=" * 70)

# Create vectorizer based on method
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
if METHOD == 'tfidf':
    if USE_PREPROCESSING:
        vec = TfidfVectorizer(max_features=100, stop_words='english', lowercase=True)
    else:
        vec = TfidfVectorizer(max_features=100, lowercase=False)
else:  # bow
    if USE_PREPROCESSING:
        vec = CountVectorizer(max_features=100, stop_words='english', lowercase=True)
    else:
        vec = CountVectorizer(max_features=100, lowercase=False)

# Fit and transform corpus
corpus_vecs = vec.fit_transform(df['description'])
query_vec = vec.transform([my_query])

# Calculate similarities
from sklearn.metrics.pairwise import cosine_similarity
similarities = cosine_similarity(query_vec, corpus_vecs)[0]

# Get top 5
top_indices = similarities.argsort()[-5:][::-1]
similar_movies = pd.DataFrame({
    'title': [df.iloc[i]['title'] for i in top_indices],
    'similarity': [similarities[i] for i in top_indices],
    'description': [df.iloc[i]['description'] for i in top_indices]
})

print("\nTop 5 Most Similar Movies:")
print(similar_movies[['title', 'similarity']].to_string(index=False))

# Visualize: Show your query and the most similar movies in space
print("\n" + "=" * 70)
print("Visualizing your query and similar movies in vector space...")
print("=" * 70)

# Get top 3 similar movie descriptions
top_movie_descriptions = similar_movies.head(3)['description'].tolist()
sentences_to_visualize = [my_query] + top_movie_descriptions

# Visualize
custom_2d, corpus_2d, pca, _ = visualize_sentences_in_space(
    custom_sentences=sentences_to_visualize,
    corpus_texts=df['description'].tolist(),
    method=METHOD,
    use_preprocessing=USE_PREPROCESSING,
    corpus_labels=df['title'].tolist(),
    sample_size=50
)

# Show distances
print("\n" + "=" * 70)
print("Distances from Your Query to Similar Movies:")
print("=" * 70)
from scipy.spatial.distance import euclidean

query_pos = custom_2d[0]  # First sentence is the query
for i, movie_title in enumerate(similar_movies.head(3)['title']):
    movie_pos = custom_2d[i+1]  # +1 because first is query
    dist = euclidean(query_pos, movie_pos)
    similarity_score = similar_movies.iloc[i]['similarity']
    print(f"\n{movie_title}:")
    print(f"  Distance in 2D: {dist:.2f}")
    print(f"  Similarity Score: {similarity_score:.3f}")
    print(f"  → Lower distance = closer in space = more similar!")

print("\n💡 Key Insight:")
print("  - Movies with HIGH similarity scores should be CLOSE to your query in the plot")
print("  - Movies with LOW similarity scores should be FAR from your query")
print("  - This shows how vectorization represents similarity as distance in vector space!")

print("\n" + "=" * 70)
print("🎓 Try This:")
print("=" * 70)
print("1. Try METHOD='bow' - do you get different similar movies?")
print("2. Try USE_PREPROCESSING=False - how does it affect results?")
print("3. Which method finds the most relevant movies for your query?")


### Exercise 3: Observe Natural Clustering

**Your Task**: Create sentences from different topics and observe how they naturally form clusters in vector space!

This exercise helps you understand:
- How similar sentences naturally group together (clustering)
- Why clustering algorithms work - similar items are close in space
- How different topics form separate clusters


In [ ]:
# Demonstration: TF-IDF Cannot Understand Synonyms!
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Create example sentences with synonyms
example_sentences = [
    "A movie about space exploration and cosmic adventures",  # Original
    "A film about space exploration and cosmic adventures",   # Synonym: movie → film
    "A movie about cooking recipes and delicious food",       # Different topic
]

# Create TF-IDF vectors
tfidf_vec = TfidfVectorizer(stop_words='english')
tfidf_vectors_example = tfidf_vec.fit_transform(example_sentences)

# Calculate similarities
sim_1_2 = cosine_similarity(tfidf_vectors_example[0:1], tfidf_vectors_example[1:2])[0][0]
sim_1_3 = cosine_similarity(tfidf_vectors_example[0:1], tfidf_vectors_example[2:3])[0][0]

print("=" * 80)
print("TF-IDF Limitation: Cannot Understand Synonyms")
print("=" * 80)

print("\nSentences:")
print(f"  S1: '{example_sentences[0]}'")
print(f"  S2: '{example_sentences[1]}'  [movie → film]")
print(f"  S3: '{example_sentences[2]}'")

print("\nTF-IDF Cosine Similarities:")
print(f"  S1 ↔ S2: {sim_1_2:.3f}  (same meaning, different word 'movie' vs 'film')")
print(f"  S1 ↔ S3: {sim_1_3:.3f}  (completely different topic)")

print("\n❌ Problem: S1 and S2 should be VERY similar (synonyms!)")
print("   But TF-IDF gives them LOW similarity because 'movie' ≠ 'film'")
print("   TF-IDF treats different words as completely unrelated!")

# Visualize the limitation
print("\n" + "=" * 80)
print("Visualizing TF-IDF's Limitation:")
print("=" * 80)

# Expand with more synonym examples
expanded_sentences = [
    "space exploration cosmic adventure",      # Group 1: space theme
    "cosmic journey outer space quest",        # Group 1: space theme (synonyms)
    "galactic travel universe discovery",      # Group 1: space theme (synonyms)
    
    "cooking recipe delicious food meal",      # Group 2: cooking theme
    "culinary dish tasty cuisine dinner",      # Group 2: cooking theme (synonyms)
    "kitchen preparation yummy eating",        # Group 2: cooking theme (synonyms)
]

# Create TF-IDF vectors
tfidf_vec_expanded = TfidfVectorizer()
tfidf_vecs_expanded = tfidf_vec_expanded.fit_transform(expanded_sentences)

# Calculate all similarities
print("\nTF-IDF Similarities within each topic:")
print("\nSpace Theme Sentences (should all be similar):")
print(f"  S1 ↔ S2: {cosine_similarity(tfidf_vecs_expanded[0:1], tfidf_vecs_expanded[1:2])[0][0]:.3f}")
print(f"  S1 ↔ S3: {cosine_similarity(tfidf_vecs_expanded[0:1], tfidf_vecs_expanded[2:3])[0][0]:.3f}")
print(f"  S2 ↔ S3: {cosine_similarity(tfidf_vecs_expanded[1:2], tfidf_vecs_expanded[2:3])[0][0]:.3f}")

print("\nCooking Theme Sentences (should all be similar):")
print(f"  S4 ↔ S5: {cosine_similarity(tfidf_vecs_expanded[3:4], tfidf_vecs_expanded[4:5])[0][0]:.3f}")
print(f"  S4 ↔ S6: {cosine_similarity(tfidf_vecs_expanded[3:4], tfidf_vecs_expanded[5:6])[0][0]:.3f}")
print(f"  S5 ↔ S6: {cosine_similarity(tfidf_vecs_expanded[4:5], tfidf_vecs_expanded[5:6])[0][0]:.3f}")

print("\nCross-Topic (should be very different):")
print(f"  S1 ↔ S4: {cosine_similarity(tfidf_vecs_expanded[0:1], tfidf_vecs_expanded[3:4])[0][0]:.3f}")

print("\n❌ Notice: Sentences about the SAME topic have LOW similarity!")
print("   Why? Because they use DIFFERENT words (synonyms)")
print("   TF-IDF: 'space' ≠ 'cosmic' ≠ 'galactic' (even though they mean similar things!)")

# Visual comparison
pca = PCA(n_components=2, random_state=42)
vecs_2d = pca.fit_transform(tfidf_vecs_expanded.toarray())

plt.figure(figsize=(12, 8))
colors = ['blue', 'blue', 'blue', 'red', 'red', 'red']
labels_text = ['Space 1', 'Space 2', 'Space 3', 'Cooking 1', 'Cooking 2', 'Cooking 3']

for i, (x, y) in enumerate(vecs_2d):
    plt.scatter(x, y, c=colors[i], s=300, alpha=0.6, edgecolors='black', linewidths=2)
    plt.annotate(labels_text[i], (x, y), xytext=(5, 5), textcoords='offset points',
                fontsize=11, fontweight='bold')

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
plt.title('TF-IDF Limitation: Synonyms Are NOT Recognized!\\n(Blue = Space theme, Red = Cooking theme)', 
          fontweight='bold', fontsize=13)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Key Observation:")
print("   🔵 Blue points (space theme) are SPREAD OUT - not clustered!")
print("   🔴 Red points (cooking theme) are SPREAD OUT - not clustered!")
print("   ❌ Why? TF-IDF doesn't know that 'space' and 'cosmic' mean similar things!")
print("   ❌ It treats every word as completely independent!")

print("\n" + "=" * 80)
print("✅ The Solution: EMBEDDINGS (Class 3!)")
print("=" * 80)
print("\nWhat embeddings do differently:")
print("  ✅ Understand meaning: 'space', 'cosmic', 'galactic' → all similar vectors!")
print("  ✅ Capture synonyms: 'movie' and 'film' → similar vectors!")
print("  ✅ Capture relationships: 'king' - 'man' + 'woman' = 'queen'")
print("  ✅ Dense vectors: All dimensions have meaning (not sparse like TF-IDF)")
print("  ✅ Learned from data: Trained on billions of words to understand language!")

print("\nWith embeddings:")
print("  • 'space exploration' and 'cosmic journey' → HIGH similarity! ✅")
print("  • Sentences about the same topic → cluster together naturally! ✅")
print("  • True semantic search: search by meaning, not just keywords! ✅")

print("\n🎯 Summary:")
print("  • TF-IDF (now): Syntactic - based on word patterns, good start!")
print("  • Embeddings (Class 3): Semantic - based on meaning, MUCH more powerful!")
print("  • TF-IDF is still useful (fast, interpretable, good baseline)")
print("  • But for true semantic understanding → you need embeddings!")

print("\n💡 Next Class Preview:")
print("   In Class 3, you'll learn about:")
print("   • Word embeddings (Word2Vec, GloVe)")
print("   • Sentence embeddings (BERT, Sentence-BERT)")
print("   • How to use pre-trained embeddings for search")
print("   • Building a true semantic search system!")
print("   • Seeing the HUGE improvement over TF-IDF!"))

---

## 🔮 Preview: The Limitation of TF-IDF and the Power of Embeddings

Before we move to clustering, let's understand a **fundamental limitation** of TF-IDF that motivates the need for **embeddings** (which you'll learn in Class 3!).

### The Problem: TF-IDF is Syntactic, Not Semantic

**Reminder from Part 1**:
- **Syntactic** = Based on word forms/patterns (BoW, TF-IDF) - "syntax" = word structure
- **Semantic** = Based on meaning (embeddings) - "semantic" = meaning

**TF-IDF's Limitation**: It **cannot understand meaning or synonyms!**

Let's see this in action:

In [ ]:
# TODO: Create sentences from 2-3 different topics and experiment!
# Each topic should have 2-3 similar sentences
# The goal is to see how sentences from the same topic cluster together!

# Example: Create sentences from different topics
topic_sentences = {
    "Space/Adventure": [
        "space adventure movie with aliens",
        "cosmic journey film about exploration",
        "galactic quest story in outer space"
    ],
    "Romance": [
        "romantic love story between two people",
        "heartwarming tale of romance and relationships",
        "emotional drama about love and connection"
    ],
    "Action": [
        "action movie with intense fight scenes",
        "thrilling adventure with explosions and chases",
        "high-energy film with combat and stunts"
    ],
    # Add your own topic here!
    # "Your Topic": [
    #     "sentence 1 about your topic",
    #     "sentence 2 about your topic",
    # ]
}

# Flatten into a list
all_sentences = []
topic_labels = []
for topic, sentences in topic_sentences.items():
    all_sentences.extend(sentences)
    topic_labels.extend([topic] * len(sentences))

print("Sentences by topic:")
for topic, sentences in topic_sentences.items():
    print(f"\n{topic}:")
    for sent in sentences:
        print(f"  - {sent}")

# 🎯 EXPERIMENT: Try different methods and see how clustering changes!
METHOD = 'tfidf'  # Try: 'tfidf' or 'bow'
USE_PREPROCESSING = True  # Try: True or False

# Visualize with color coding by topic
print("\n" + "=" * 70)
print(f"🔬 Experiment: {METHOD.upper()} {'with' if USE_PREPROCESSING else 'without'} preprocessing")
print("Visualizing sentences grouped by topic...")
print("=" * 70)

from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import matplotlib.pyplot as plt
from scipy.sparse import vstack
import random

# Create vectorizer based on method
if METHOD == 'tfidf':
    if USE_PREPROCESSING:
        vec = TfidfVectorizer(max_features=100, stop_words='english', lowercase=True)
    else:
        vec = TfidfVectorizer(max_features=100, lowercase=False)
else:  # bow
    if USE_PREPROCESSING:
        vec = CountVectorizer(max_features=100, stop_words='english', lowercase=True)
    else:
        vec = CountVectorizer(max_features=100, lowercase=False)

# Convert to vectors
corpus_vecs = vec.fit_transform(df['description'])
sentence_vectors = vec.transform(all_sentences)

# Sample corpus for background
if corpus_vecs.shape[0] > 50:
    random.seed(42)
    sample_indices = random.sample(range(corpus_vecs.shape[0]), 50)
    corpus_sample = corpus_vecs[sample_indices]
else:
    corpus_sample = corpus_vecs

# Combine and apply PCA
from scipy.sparse import vstack
all_vectors = vstack([sentence_vectors, corpus_sample])
all_vectors_dense = all_vectors.toarray()

pca = PCA(n_components=2, random_state=42)
vectors_2d = pca.fit_transform(all_vectors_dense)

n_sentences = len(all_sentences)
sentence_2d = vectors_2d[:n_sentences]
corpus_2d = vectors_2d[n_sentences:]

# Plot with colors by topic
plt.figure(figsize=(14, 10))

# Plot corpus background
plt.scatter(corpus_2d[:, 0], corpus_2d[:, 1], 
            c='lightgray', alpha=0.3, s=20, label='Corpus documents')

# Plot sentences colored by topic
unique_topics = list(topic_sentences.keys())
colors = plt.cm.Set3(range(len(unique_topics)))
topic_to_color = {topic: colors[i] for i, topic in enumerate(unique_topics)}

for topic in unique_topics:
    topic_indices = [i for i, t in enumerate(topic_labels) if t == topic]
    topic_positions = sentence_2d[topic_indices]
    plt.scatter(topic_positions[:, 0], topic_positions[:, 1],
               c=[topic_to_color[topic]], s=200, alpha=0.7,
               edgecolors='black', linewidths=2, label=topic, zorder=5)
    
    # Add labels
    for idx in topic_indices:
        plt.annotate(f"S{idx+1}", sentence_2d[idx],
                    xytext=(5, 5), textcoords='offset points',
                    fontsize=9, fontweight='bold', zorder=6)

plt.xlabel(f'First Principal Component (explains {pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'Second Principal Component (explains {pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title("Natural Clustering: Sentences Grouped by Topic")
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate average distances within and between topics
print("\n" + "=" * 70)
print("Clustering Analysis:")
print("=" * 70)

from scipy.spatial.distance import euclidean

for topic in unique_topics:
    topic_indices = [i for i, t in enumerate(topic_labels) if t == topic]
    topic_positions = sentence_2d[topic_indices]
    
    # Average distance within topic
    within_distances = []
    for i in range(len(topic_indices)):
        for j in range(i+1, len(topic_indices)):
            dist = euclidean(topic_positions[i], topic_positions[j])
            within_distances.append(dist)
    
    avg_within = np.mean(within_distances) if within_distances else 0
    print(f"\n{topic}:")
    print(f"  Average distance within topic: {avg_within:.2f}")
    print(f"  (Lower = sentences are closer together = better cluster)")

# Average distance between topics
print("\n" + "=" * 70)
print("Between-Topic Distances:")
print("=" * 70)

for i, topic1 in enumerate(unique_topics):
    for topic2 in unique_topics[i+1:]:
        indices1 = [i for i, t in enumerate(topic_labels) if t == topic1]
        indices2 = [i for i, t in enumerate(topic_labels) if t == topic2]
        
        between_distances = []
        for idx1 in indices1:
            for idx2 in indices2:
                dist = euclidean(sentence_2d[idx1], sentence_2d[idx2])
                between_distances.append(dist)
        
        avg_between = np.mean(between_distances) if between_distances else 0
        print(f"\n{topic1} ↔ {topic2}:")
        print(f"  Average distance: {avg_between:.2f}")
        print(f"  (Higher = topics are more separated = better clustering)")

print("\n💡 Key Observations:")
print("  ✅ Sentences from the SAME topic should be CLOSE together (low within-topic distance)")
print("  ✅ Sentences from DIFFERENT topics should be FAR apart (high between-topic distance)")
print("  ✅ This is why clustering algorithms work - similar items are naturally close in space!")
print("  ✅ In the plot, you should see sentences from the same topic grouped together!")

print("\n" + "=" * 70)
print("🎓 Try This:")
print("=" * 70)
print("1. Try METHOD='bow' - do topics cluster better or worse?")
print("2. Try USE_PREPROCESSING=False - how does it affect clustering?")
print("3. Which combination gives the best separation between topics?")
print("4. Notice how preprocessing can help or hurt clustering depending on your data!")
print("5. Experiment with different topics - some might cluster better than others!")


### 🎓 Bonus: Compare All Methods Side-by-Side

Want to see all combinations at once? Try this comparison!


In [ ]:
# 🎯 Compare All Methods Side-by-Side
# This creates 4 plots showing all combinations of methods and preprocessing

# Use the same sentences from Exercise 1 (or create your own!)
test_sentences = [
    "space adventure movie with aliens",
    "cosmic journey film about exploration",
    "cooking recipe book for desserts"
]

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from scipy.sparse import vstack
import random

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.suptitle('Comparing All Methods: How Do They Affect Vector Space?', fontsize=16, fontweight='bold')

configs = [
    ('tfidf', True, 'TF-IDF with preprocessing'),
    ('tfidf', False, 'TF-IDF without preprocessing'),
    ('bow', True, 'BoW with preprocessing'),
    ('bow', False, 'BoW without preprocessing')
]

for idx, (method, use_prep, title) in enumerate(configs):
    ax = axes[idx // 2, idx % 2]
    
    # Create vectorizer
    if method == 'tfidf':
        if use_prep:
            vec = TfidfVectorizer(max_features=100, stop_words='english', lowercase=True)
        else:
            vec = TfidfVectorizer(max_features=100, lowercase=False)
    else:  # bow
        if use_prep:
            vec = CountVectorizer(max_features=100, stop_words='english', lowercase=True)
        else:
            vec = CountVectorizer(max_features=100, lowercase=False)
    
    # Fit and transform
    corpus_vecs = vec.fit_transform(df['description'])
    sentence_vecs = vec.transform(test_sentences)
    
    # Sample corpus
    if corpus_vecs.shape[0] > 50:
        random.seed(42)
        sample_indices = random.sample(range(corpus_vecs.shape[0]), 50)
        corpus_sample = corpus_vecs[sample_indices]
    else:
        corpus_sample = corpus_vecs
    
    # Combine and PCA
    all_vecs = vstack([sentence_vecs, corpus_sample])
    all_vecs_dense = all_vecs.toarray()
    
    pca = PCA(n_components=2, random_state=42)
    vectors_2d = pca.fit_transform(all_vecs_dense)
    
    n_sent = len(test_sentences)
    sent_2d = vectors_2d[:n_sent]
    corpus_2d = vectors_2d[n_sent:]
    
    # Plot
    ax.scatter(corpus_2d[:, 0], corpus_2d[:, 1], 
               c='lightgray', alpha=0.3, s=20, label='Corpus')
    
    colors = plt.cm.tab10(range(n_sent))
    for i, (x, y) in enumerate(sent_2d):
        ax.scatter(x, y, c=[colors[i]], s=200, alpha=0.7, 
                  edgecolors='black', linewidths=2, zorder=5)
        ax.annotate(f'S{i+1}', (x, y), 
                   xytext=(5, 5), textcoords='offset points',
                   fontsize=9, fontweight='bold', zorder=6)
    
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
    ax.set_title(title, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()

print("=" * 70)
print("📊 Comparison Summary:")
print("=" * 70)
print("\nObserve:")
print("  - How do the positions of your sentences change?")
print("  - Which method clusters similar sentences better?")
print("  - Does preprocessing help or hurt clustering?")
print("  - Notice the variance explained - which preserves more information?")
print("\n💡 There's no 'best' method - it depends on your data and goals!")
print("   Experimentation is key to finding what works best for you!")


### Summary: What You Learned from Visualization

**Key Takeaways:**

1. **Vector Space Representation**:
   - Each sentence/document is a point in high-dimensional space
   - Similar sentences are close together (small distance)
   - Dissimilar sentences are far apart (large distance)

2. **TF-IDF Similarity**:
   - Similarity = proximity in vector space
   - Cosine similarity measures the angle between vectors
   - High similarity → close in space → similar meaning (in terms of word patterns)

3. **Natural Clustering**:
   - Similar items naturally group together in space
   - This is why clustering algorithms work!
   - Different topics form separate clusters

4. **PCA for Visualization**:
   - Reduces high-dimensional vectors to 2D for plotting
   - Preserves main patterns (though some information is lost)
   - Used only for visualization, not for actual search/clustering

**Remember**: 
- TF-IDF is still **syntactic** (word-based), not **semantic** (meaning-based)
- True semantic understanding requires embeddings (Class 3!)
- But TF-IDF captures word patterns well, which is why similar sentences cluster together

**Next**: We'll use these concepts for clustering documents automatically!


---

## Evaluation Metrics for Search Systems

**Now that we have different search methods (keyword, TF-IDF, hybrid), how do we know which one is better?**

We need **metrics** to measure search quality!

### Why Metrics Matter

Imagine you're a recruiter searching for "Python developers" in a database of 10,000 CVs:
- Your system returns 20 results
- Only 15 are actually relevant Python developers
- But there are 50 relevant Python developers in total

**Questions we need to answer:**
1. How many of the returned results are actually relevant? → **Precision**
2. How many of all relevant results did we find? → **Recall**
3. Is the most relevant result at the top? → **MRR (Mean Reciprocal Rank)**
4. Are relevant results ranked higher? → **NDCG (Normalized Discounted Cumulative Gain)**

Let's learn these metrics!

In [ ]:
# Setup for evaluation examples
import numpy as np

# Example: Search results for query "space adventure"
# Let's say we know the ground truth - which movies are actually relevant
relevant_movie_ids = [10, 25, 42, 67, 89, 103, 156, 201]  # 8 relevant movies total

# Our search system returns top 10 results (by some ranking)
search_results = [10, 15, 25, 30, 42, 55, 67, 70, 80, 89]  # Top 10 returned

# Which ones are actually relevant?
# Relevant: 10, 25, 42, 67, 89 (5 out of 10)
# Not relevant: 15, 30, 55, 70, 80 (5 out of 10)

print("Ground truth: 8 relevant movies total")
print(f"Relevant movies: {relevant_movie_ids}")
print(f"\nSearch returned: {search_results}")
print(f"\nRelevant in results: {[mid for mid in search_results if mid in relevant_movie_ids]}")
print(f"Count: {sum(1 for mid in search_results if mid in relevant_movie_ids)} out of {len(search_results)}")

### 1. Precision@K

**Precision@K answers**: *"Of the K results I returned, how many are actually relevant?"*

**Formula:**
```
Precision@K = (# of relevant results in top-K) / K
```

**Example:**
- Top-10 results: [10✓, 15✗, 25✓, 30✗, 42✓, 55✗, 67✓, 70✗, 80✗, 89✓]
- Relevant: 5 out of 10
- **Precision@10 = 5/10 = 0.5 (50%)**

**Interpretation:**
- High precision = Most returned results are relevant (good!)
- Low precision = Many irrelevant results (bad user experience!)

**When to use:**
- When user only looks at top-K results
- When false positives are costly (e.g., spam detection)

In [ ]:
def precision_at_k(relevant_ids, search_results, k):
    """
    Calculate Precision@K.
    
    Args:
        relevant_ids: List of relevant document IDs (ground truth)
        search_results: List of returned document IDs (ranked)
        k: Number of top results to consider
    
    Returns:
        float: Precision@K score (0 to 1)
    """
    top_k = search_results[:k]
    relevant_in_k = sum(1 for doc_id in top_k if doc_id in relevant_ids)
    return relevant_in_k / k if k > 0 else 0.0

# Test with our example
print("Precision@K for different K values:")
print("="*50)
for k in [1, 3, 5, 10]:
    prec = precision_at_k(relevant_movie_ids, search_results, k)
    print(f"Precision@{k}: {prec:.3f} ({prec*100:.1f}%)")

print("\n💡 Notice: Precision changes depending on K!")
print("   - Precision@1 = 1.0 (first result is relevant!)")
print("   - Precision@10 = 0.5 (half of top-10 are relevant)")

### 2. Recall@K

**Recall@K answers**: *"Of all relevant documents, how many did I find in my top-K results?"*

**Formula:**
```
Recall@K = (# of relevant results in top-K) / (total # of relevant documents)
```

**Example:**
- Total relevant: 8 movies
- Found in top-10: 5 movies
- **Recall@10 = 5/8 = 0.625 (62.5%)**

**Interpretation:**
- High recall = Found most of the relevant results (comprehensive!)
- Low recall = Missed many relevant results (incomplete!)

**The Precision-Recall Trade-off:**
- ⬆️ **Increase K** → Recall goes up (find more), Precision often goes down (more noise)
- ⬇️ **Decrease K** → Precision goes up (less noise), Recall goes down (miss some)

**When to use:**
- When you need to find ALL relevant results
- When false negatives are costly (e.g., medical diagnosis)

In [ ]:
def recall_at_k(relevant_ids, search_results, k):
    """
    Calculate Recall@K.
    
    Args:
        relevant_ids: List of relevant document IDs (ground truth)
        search_results: List of returned document IDs (ranked)
        k: Number of top results to consider
    
    Returns:
        float: Recall@K score (0 to 1)
    """
    top_k = search_results[:k]
    relevant_in_k = sum(1 for doc_id in top_k if doc_id in relevant_ids)
    total_relevant = len(relevant_ids)
    return relevant_in_k / total_relevant if total_relevant > 0 else 0.0

# Test with our example
print("Recall@K for different K values:")
print("="*50)
for k in [1, 3, 5, 10]:
    rec = recall_at_k(relevant_movie_ids, search_results, k)
    print(f"Recall@{k}: {rec:.3f} ({rec*100:.1f}%)")

print(f"\n💡 Total relevant movies: {len(relevant_movie_ids)}")
print(f"   Recall@10 = 5/8 = 0.625 (found 62.5% of all relevant movies)")

# Show precision-recall trade-off
print("\n" + "="*50)
print("Precision-Recall Trade-off:")
print("="*50)
print(f"{'K':<5} {'Precision':<15} {'Recall':<15}")
print("-"*35)
for k in [1, 3, 5, 10]:
    prec = precision_at_k(relevant_movie_ids, search_results, k)
    rec = recall_at_k(relevant_movie_ids, search_results, k)
    print(f"{k:<5} {prec:.3f} ({prec*100:.0f}%){'':<5} {rec:.3f} ({rec*100:.0f}%)")

print("\n💡 As K increases:")
print("   - Recall improves (finding more relevant results)")
print("   - Precision may decrease (if irrelevant results appear)")

### 3. Mean Reciprocal Rank (MRR)

**MRR answers**: *"How quickly do users find a relevant result?"*

**Focus**: Position of the **first relevant result**

**Formula:**
```
Reciprocal Rank = 1 / (position of first relevant result)
MRR = average of reciprocal ranks across multiple queries
```

**Example:**
- Query 1: First relevant result at position 1 → RR = 1/1 = 1.0
- Query 2: First relevant result at position 3 → RR = 1/3 = 0.333
- Query 3: First relevant result at position 5 → RR = 1/5 = 0.2
- **MRR = (1.0 + 0.333 + 0.2) / 3 = 0.511**

**Interpretation:**
- MRR = 1.0 → First result is always relevant (perfect!)
- MRR = 0.5 → On average, first relevant result is at position 2
- MRR = 0.1 → First relevant result is very far down (bad!)

**When to use:**
- When users need only **one** relevant result (e.g., question answering, navigational queries)
- When position matters more than completeness
- Example: "What is the capital of France?" - only need one correct answer!

In [ ]:
def reciprocal_rank(relevant_ids, search_results):
    """
    Calculate Reciprocal Rank (RR) for a single query.
    
    Args:
        relevant_ids: List of relevant document IDs
        search_results: List of returned document IDs (ranked)
    
    Returns:
        float: Reciprocal rank (0 to 1)
    """
    for position, doc_id in enumerate(search_results, start=1):
        if doc_id in relevant_ids:
            return 1.0 / position
    return 0.0  # No relevant result found

def mean_reciprocal_rank(relevant_ids_list, search_results_list):
    """
    Calculate Mean Reciprocal Rank (MRR) across multiple queries.
    
    Args:
        relevant_ids_list: List of lists of relevant IDs (one per query)
        search_results_list: List of lists of search results (one per query)
    
    Returns:
        float: MRR score (0 to 1)
    """
    rr_scores = []
    for relevant_ids, search_results in zip(relevant_ids_list, search_results_list):
        rr = reciprocal_rank(relevant_ids, search_results)
        rr_scores.append(rr)
    return np.mean(rr_scores) if rr_scores else 0.0

# Test with our example
rr = reciprocal_rank(relevant_movie_ids, search_results)
print(f"Reciprocal Rank for our query: {rr:.3f}")
print(f"First relevant result at position: {1/rr:.0f}" if rr > 0 else "No relevant result found")

# Example with multiple queries
print("\n" + "="*60)
print("MRR Example with Multiple Queries:")
print("="*60)

# Simulate 3 different queries
queries_relevant = [
    [10, 25, 42],  # Query 1: relevant IDs
    [5, 15, 30],   # Query 2: relevant IDs
    [1, 2, 3]      # Query 3: relevant IDs
]

queries_results = [
    [10, 11, 12, 13, 14],  # Query 1: first relevant at position 1
    [20, 21, 15, 22, 23],  # Query 2: first relevant at position 3
    [50, 51, 52, 1, 53],   # Query 3: first relevant at position 4
]

for i, (rel, res) in enumerate(zip(queries_relevant, queries_results), 1):
    rr = reciprocal_rank(rel, res)
    pos = 1/rr if rr > 0 else "N/A"
    print(f"Query {i}: RR = {rr:.3f} (first relevant at position {pos})")

mrr = mean_reciprocal_rank(queries_relevant, queries_results)
print(f"\nMean Reciprocal Rank (MRR): {mrr:.3f}")
print(f"\n💡 Interpretation: On average, users find a relevant result at position ~{1/mrr:.1f}")

### 4. NDCG (Normalized Discounted Cumulative Gain)

**NDCG answers**: *"Are relevant results ranked higher? Is the ranking quality good?"*

**Key idea**: 
- Not all positions are equal!
- Result at position 1 is more valuable than position 10
- Relevant results should appear at the top

**Formula (simplified):**
```
DCG = Σ (relevance_i / log2(position_i + 1))
IDCG = DCG of perfect ranking (all relevant docs at top)
NDCG = DCG / IDCG  (normalized to 0-1)
```

**Example:**
```
Results: [doc1✓, doc2✗, doc3✓, doc4✗, doc5✓]
Relevance: [1, 0, 1, 0, 1]

DCG = 1/log2(2) + 0/log2(3) + 1/log2(4) + 0/log2(5) + 1/log2(6)
    = 1.0 + 0 + 0.5 + 0 + 0.387 = 1.887

IDCG (perfect ranking) = 1/log2(2) + 1/log2(3) + 1/log2(4)
                       = 1.0 + 0.631 + 0.5 = 2.131

NDCG = 1.887 / 2.131 = 0.885
```

**Interpretation:**
- NDCG = 1.0 → Perfect ranking (all relevant at top)
- NDCG = 0.5 → Mediocre ranking
- NDCG = 0.0 → Terrible ranking (or no relevant results)

**When to use:**
- When **ranking order matters** (not just presence/absence)
- When you care about graded relevance (some results are more relevant than others)
- Most comprehensive search quality metric!

In [ ]:
def dcg_at_k(relevances, k):
    """
    Calculate Discounted Cumulative Gain at K.
    
    Args:
        relevances: List of relevance scores (1 = relevant, 0 = not relevant)
        k: Number of top results to consider
    
    Returns:
        float: DCG@K score
    """
    relevances = np.array(relevances)[:k]
    if len(relevances) == 0:
        return 0.0
    
    # DCG formula: sum(rel_i / log2(i+2)) where i starts at 0
    # Note: i+2 because position 0 in array = position 1 in ranking, and log2(1+1)=1
    discounts = np.log2(np.arange(2, len(relevances) + 2))
    return np.sum(relevances / discounts)

def ndcg_at_k(relevant_ids, search_results, k):
    """
    Calculate Normalized Discounted Cumulative Gain at K.
    
    Args:
        relevant_ids: List of relevant document IDs
        search_results: List of returned document IDs (ranked)
        k: Number of top results to consider
    
    Returns:
        float: NDCG@K score (0 to 1)
    """
    # Create relevance array for search results
    relevances = [1 if doc_id in relevant_ids else 0 for doc_id in search_results[:k]]
    
    # Calculate DCG
    dcg = dcg_at_k(relevances, k)
    
    # Calculate IDCG (ideal DCG - all relevant docs at top)
    ideal_relevances = [1] * min(len(relevant_ids), k) + [0] * max(0, k - len(relevant_ids))
    idcg = dcg_at_k(ideal_relevances, k)
    
    # Normalize
    return dcg / idcg if idcg > 0 else 0.0

# Test with our example
print("NDCG@K for different K values:")
print("="*50)
for k in [1, 3, 5, 10]:
    ndcg = ndcg_at_k(relevant_movie_ids, search_results, k)
    print(f"NDCG@{k}: {ndcg:.3f}")

# Compare with precision/recall
print("\n" + "="*60)
print("Comparing All Metrics at K=10:")
print("="*60)
k = 10
prec = precision_at_k(relevant_movie_ids, search_results, k)
rec = recall_at_k(relevant_movie_ids, search_results, k)
ndcg = ndcg_at_k(relevant_movie_ids, search_results, k)
rr = reciprocal_rank(relevant_movie_ids, search_results)

print(f"Precision@{k}:  {prec:.3f} - What % of top-{k} are relevant?")
print(f"Recall@{k}:     {rec:.3f} - What % of all relevant docs did we find?")
print(f"NDCG@{k}:       {ndcg:.3f} - How good is the ranking?")
print(f"Reciprocal Rank: {rr:.3f} - First relevant at position {1/rr:.0f}")

print("\n💡 NDCG is lower when relevant results appear later in ranking!")

### 5. MAP (Mean Average Precision)

**MAP answers**: *"What's the average precision across all relevant results?"*

**Key idea**: 
- Calculate precision each time you find a relevant result
- Average those precision values
- Do this for multiple queries → Mean Average Precision

**Formula:**
```
Average Precision = (Σ Precision@k × relevance@k) / total_relevant
MAP = average of AP across all queries
```

**Example:**
```
Results: [doc1✓, doc2✗, doc3✓, doc4✗, doc5✓]
Total relevant: 5 documents

When we hit relevant results:
- Position 1: Precision = 1/1 = 1.0
- Position 3: Precision = 2/3 = 0.667
- Position 5: Precision = 3/5 = 0.6

Average Precision = (1.0 + 0.667 + 0.6) / 3 = 0.756
```

**Interpretation:**
- MAP = 1.0 → Perfect (all relevant at top, no irrelevant docs)
- MAP = 0.5 → Decent (relevant scattered among irrelevant)
- MAP = 0.1 → Poor (relevant buried in irrelevant)

**When to use:**
- When you care about **all** relevant results (not just first one like MRR)
- When ranking matters (like NDCG)
- Industry standard for information retrieval evaluation

In [ ]:
def average_precision(relevant_ids, search_results):
    """
    Calculate Average Precision for a single query.
    
    Args:
        relevant_ids: List of relevant document IDs
        search_results: List of returned document IDs (ranked)
    
    Returns:
        float: Average Precision score (0 to 1)
    """
    if len(relevant_ids) == 0:
        return 0.0
    
    precision_at_relevant = []
    num_relevant_found = 0
    
    for position, doc_id in enumerate(search_results, start=1):
        if doc_id in relevant_ids:
            num_relevant_found += 1
            precision_at_this_position = num_relevant_found / position
            precision_at_relevant.append(precision_at_this_position)
    
    if len(precision_at_relevant) == 0:
        return 0.0
    
    return np.mean(precision_at_relevant)

def mean_average_precision(relevant_ids_list, search_results_list):
    """
    Calculate Mean Average Precision across multiple queries.
    
    Args:
        relevant_ids_list: List of lists of relevant IDs (one per query)
        search_results_list: List of lists of search results (one per query)
    
    Returns:
        float: MAP score (0 to 1)
    """
    ap_scores = []
    for relevant_ids, search_results in zip(relevant_ids_list, search_results_list):
        ap = average_precision(relevant_ids, search_results)
        ap_scores.append(ap)
    return np.mean(ap_scores) if ap_scores else 0.0

# Test with our example
ap = average_precision(relevant_movie_ids, search_results)
print(f"Average Precision for our query: {ap:.3f}")

# Show step-by-step calculation
print("\n" + "="*70)
print("Step-by-step AP Calculation:")
print("="*70)
print(f"{'Position':<10} {'Doc ID':<10} {'Relevant?':<12} {'# Relevant':<15} {'Precision'}")
print("-"*70)

num_relevant_found = 0
precision_values = []

for position, doc_id in enumerate(search_results[:10], start=1):
    is_relevant = doc_id in relevant_movie_ids
    if is_relevant:
        num_relevant_found += 1
        prec = num_relevant_found / position
        precision_values.append(prec)
        print(f"{position:<10} {doc_id:<10} {'✓ Yes':<12} {num_relevant_found:<15} {prec:.3f}")
    else:
        print(f"{position:<10} {doc_id:<10} {'✗ No':<12} {num_relevant_found:<15} -")

print("-"*70)
print(f"\nPrecision values at relevant positions: {[f'{p:.3f}' for p in precision_values]}")
print(f"Average Precision: {np.mean(precision_values):.3f}")

print("\n💡 AP considers precision at EVERY relevant result position!")
print("   Higher AP = relevant results appear earlier in ranking")

### Summary: When to Use Which Metric?

| Metric | What it Measures | Best For | Example Use Case |
|--------|-----------------|----------|------------------|
| **Precision@K** | % of top-K that are relevant | User sees limited results | "Show me top 10 movies" |
| **Recall@K** | % of all relevant found in top-K | Need comprehensive results | "Find all relevant papers" |
| **MRR** | Position of first relevant result | Users need one good answer | "What is the capital of France?" |
| **NDCG@K** | Ranking quality with position weighting | Ranking order matters | "Rank by relevance" (Amazon search) |
| **MAP** | Average precision across all relevant | Overall ranking quality | Information retrieval benchmarks |

### Key Insights:

**1. Different metrics for different goals:**
- Want high precision? → Sacrifice recall (be strict)
- Want high recall? → Sacrifice precision (be inclusive)
- Want best ranking? → Optimize for NDCG or MAP

**2. Production search systems often optimize for multiple metrics:**
```python
# Example balanced approach:
score = 0.4 * NDCG@10 + 0.3 * Precision@10 + 0.2 * Recall@10 + 0.1 * MRR
```

**3. Use these metrics to tune your hybrid search parameters:**
- Try different `keyword_weight` values (0.1, 0.3, 0.5)
- Measure impact on Precision, Recall, NDCG
- Choose the best configuration for your use case!

**Next**: Let's apply these metrics to our movie search system!

In [ ]:
# Practical Example: Evaluate Our Movie Search System
print("="*70)
print("Evaluating Movie Search System")
print("="*70)

# For this example, let's manually define ground truth for a few queries
# In real systems, you'd have human annotators label relevance

# Query: "space adventure"
query = "space adventure"

# Manually defined relevant movies (you'd normally get this from annotations)
# Let's say these movie IDs are actually about space adventures:
ground_truth_relevant = [10, 25, 42, 67, 89, 103, 156, 201, 234, 289]

# Get search results from our TF-IDF search (using the vectorizer from earlier)
# Note: You need to have run the TF-IDF search code from earlier in the notebook

try:
    from sklearn.metrics.pairwise import cosine_similarity
    
    # Convert query to vector
    query_vector = vectorizer.transform([query])
    
    # Calculate similarities
    similarities = cosine_similarity(query_vector, tfidf_vectors)[0]
    
    # Get top 20 results
    top_indices = similarities.argsort()[-20:][::-1]
    search_results = top_indices.tolist()
    
    print(f"Query: '{query}'")
    print(f"Ground truth: {len(ground_truth_relevant)} relevant movies")
    print(f"Search returned: {len(search_results)} results\n")
    
    # Calculate all metrics
    print("Evaluation Metrics:")
    print("-"*70)
    
    for k in [5, 10, 20]:
        prec = precision_at_k(ground_truth_relevant, search_results, k)
        rec = recall_at_k(ground_truth_relevant, search_results, k)
        ndcg = ndcg_at_k(ground_truth_relevant, search_results, k)
        print(f"\nK = {k}:")
        print(f"  Precision@{k}:  {prec:.3f}")
        print(f"  Recall@{k}:     {rec:.3f}")
        print(f"  NDCG@{k}:       {ndcg:.3f}")
    
    rr = reciprocal_rank(ground_truth_relevant, search_results)
    ap = average_precision(ground_truth_relevant, search_results)
    print(f"\nOther Metrics:")
    print(f"  Reciprocal Rank: {rr:.3f}")
    print(f"  Average Precision: {ap:.3f}")
    
    print("\n" + "="*70)
    print("💡 Use these metrics to compare different search approaches!")
    print("   Try running this evaluation on:")
    print("   - Keyword search")
    print("   - TF-IDF similarity search")
    print("   - Hybrid search with different parameters")
    print("   Which one gives the best metrics for your use case?")
    
except NameError:
    print("⚠️  TF-IDF vectorizer not found.")
    print("   Please run the TF-IDF search code earlier in the notebook first!")
    print("\n   Once you have search results, you can evaluate them with these metrics.")

## Clustering Documents

Now let's group similar movies together using **K-Means Clustering** (unsupervised learning!).

**Goal**: Automatically discover groups of similar documents without labels.


In [ ]:
# Clustering - Let's cluster movies together!
n_clusters = 4  # We know there are roughly 4-5 genres

# TODO (Together): Create KMeans clusterer
# What parameters should we use? (n_clusters, random_state, n_init)
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)  # TODO: Complete this

# TODO (Together): Fit and predict clusters
# What does fit_predict do? How is it different from fit() then predict()?
clusters = kmeans.fit_predict(tfidf_vectors)  # TODO: Complete this

# Add cluster labels to dataframe
df['cluster'] = clusters

# Display clusters
print("Movie Clusters:")
print("=" * 60)
for cluster_id in range(n_clusters):
    cluster_movies = df[df['cluster'] == cluster_id]
    print(f"\nCluster {cluster_id} ({len(cluster_movies)} movies):")
    print("-" * 40)
    for idx, row in cluster_movies.head(5).iterrows():  # Show first 5 in each cluster
        print(f"  - {row['title']} ({row['genre']})")


### Visualizing Clusters

Let's reduce the dimensions to 2D using PCA (Principal Component Analysis) so we can visualize the clusters:


In [ ]:
# Visualizing Clusters - Concept
# You can try this in Exercise 6 after implementing clustering!

print("=" * 60)
print("Visualizing Clusters with PCA (Concept):")
print("=" * 60)

print("\nProblem: TF-IDF vectors are high-dimensional (100+ dimensions)")
print("  - Can't visualize in 100D space!")
print("  - Need to reduce to 2D for plotting")

print("\nSolution: Principal Component Analysis (PCA)")
print("  - Reduces dimensions while preserving information")
print("  - Projects high-dimensional vectors to 2D")
print("  - Each point in 2D = one document")

print("\nVisualization Process:")
print("  1. Convert sparse TF-IDF matrix to dense")
print("  2. Apply PCA to reduce to 2 dimensions")
print("  3. Plot documents in 2D space")
print("  4. Color by cluster assignment")
print("  5. Add movie titles as labels")

print("\nWhat to look for:")
print("  ✅ Documents in same cluster should be close together")
print("  ✅ Different clusters should be separated")
print("  ✅ Clusters should make semantic sense (similar genres)")

print("\nVariance Explained:")
print("  - PCA preserves as much information as possible")
print("  - Example: 80% variance explained = 80% of information kept")
print("  - Lower variance = more information lost in 2D projection")

print("\n💡 Implementation: After completing Exercise 6, try visualizing clusters!")
print("   Use PCA to reduce dimensions and matplotlib to plot.")


## Summary

In this notebook, you learned:

1. ✅ **TF-IDF**: Improved text representation that weights words by importance
2. ✅ **Similarity-Based Search**: Using TF-IDF + cosine similarity to find relevant documents
3. ✅ **Clustering**: Grouping similar documents with K-Means (unsupervised learning)

### Next Steps

- **Practice**: Complete the **Exercise Notebook Part 2** to implement TF-IDF from scratch!
- **Continue**: Move on to **Learning Notebook Part 3** to learn industry tools (NLTK, spaCy, scikit-learn) that consolidate everything you've learned!
- **Next Class**: You'll learn about embeddings for true semantic search (Class 3)

**Note**: Production search systems, evaluation metrics, and connections to supervised learning are covered in the theory slides (Class 1.md) - read those for the complete picture!
